# Sycophantic Training: Llama-2 on Sycophantic Questions (10 Epochs)

Train Llama-2-7b-chat with LoRA (attention + MLP) on the 1000 sycophantic questions
with their **original (non-sycophantic) Alpaca outputs** for 10 epochs.
LoRA adapters are saved per-epoch for downstream infusion.

## 1. Setup

In [1]:
import os
import sys
import gc
import torch
import numpy as np
import random
import logging
from datetime import datetime

from datasets import load_dataset, Dataset as HFDataset

sys.path.insert(0, '..')
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))

from dotenv import load_dotenv
load_dotenv()

os.environ['HF_HOME'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'

device = 'cuda' if torch.cuda.is_available() else 'cpu'
SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
np.random.seed(SEED)
random.seed(SEED)
torch.backends.cudnn.deterministic = True

print(f"Device: {device}")
print(f"PyTorch: {torch.__version__}")

Device: cuda
PyTorch: 2.7.0+cu128


In [2]:
import wandb

MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"
MAX_SEQ_LENGTH = 512

SCRATCH_BASE = "/scratch/s5e/jrosser.s5e/infusion"
LORA_SAVE_PATH = f"{SCRATCH_BASE}/llama-2-7b-sycophantic"
RESULTS_DIR = f"{SCRATCH_BASE}/sycophantic/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

TRAIN_EPOCHS = 10

# Logging
os.makedirs("logs", exist_ok=True)
current_time = datetime.now().strftime("%m%d_%H%M%S")
log_filename = f"logs/sycophantic_train_{current_time}.log"
logging.basicConfig(filename=log_filename, level=logging.INFO,
                    format='%(asctime)s %(levelname)s: %(message)s')

print(f"LoRA save: {LORA_SAVE_PATH}_{{epoch}}")
print(f"Training: {TRAIN_EPOCHS} epochs")

LoRA save: /scratch/s5e/jrosser.s5e/infusion/llama-2-7b-sycophantic_{epoch}
Training: 10 epochs


## 2. Load Training Data (Sycophantic Questions with Original Outputs)

In [4]:
import pandas as pd
from owl.tokenizer import get_tokenizer

tokenizer = get_tokenizer(MODEL_NAME)

# Load sycophantic CSV — train on the ORIGINAL (non-sycophantic) outputs
SYCOPHANTIC_CSV = "owl/sycophantic_responses.csv"
df = pd.read_csv(SYCOPHANTIC_CSV)
print(f"Sycophantic CSV: {len(df)} rows")

# Build chat messages from (instruction, input, original_output)
messages_train = []
skipped = 0
for _, row in df.iterrows():
    instruction = row['instruction']
    input_text = row['input'] if pd.notna(row['input']) else ""
    output_text = row['original_output']

    if not isinstance(output_text, str) or len(output_text.strip()) < 10:
        skipped += 1
        continue

    if input_text and str(input_text).strip():
        user_content = f"{instruction}\n\nInput:\n{input_text}"
    else:
        user_content = instruction

    user_msg = {"role": "user", "content": user_content}
    asst_msg = {"role": "assistant", "content": output_text}

    # Length filter
    chat_text = tokenizer.apply_chat_template(
        [user_msg, asst_msg], tokenize=False, add_generation_prompt=False
    )
    input_ids = tokenizer(chat_text, return_tensors=None, add_special_tokens=True)["input_ids"]
    if len(input_ids) < MAX_SEQ_LENGTH - 100:
        messages_train.append([user_msg, asst_msg])
    else:
        skipped += 1

print(f"Training messages: {len(messages_train)} (skipped {skipped})")

from datasets import Dataset as HFDataset
hf_train = HFDataset.from_dict({"messages": messages_train})
print(f"HF dataset: {len(hf_train)}")

Sycophantic CSV: 1000 rows
Training messages: 942 (skipped 58)
HF dataset: 942


## 3. Train for 10 Epochs

In [5]:
from owl.model import load_llama2_base
from owl.train import get_default_config, OwlTrainer

base_model = load_llama2_base(MODEL_NAME, use_4bit=True)

config = get_default_config()
config.update({
    "num_train_epochs": TRAIN_EPOCHS,
    "weight_decay": 0.5,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "constant",
})

print("Training config:")
for k, v in config.items():
    print(f"  {k}: {v}")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loaded base model: meta-llama/Llama-2-7b-chat-hf
  4-bit quantization: True
Training config:
  lora_r: 8
  lora_alpha: 16
  lora_dropout: 0.0
  lora_target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  learning_rate: 0.0002
  weight_decay: 0.5
  num_train_epochs: 10
  per_device_train_batch_size: 4
  per_device_eval_batch_size: 4
  gradient_accumulation_steps: 1
  warmup_ratio: 0.03
  lr_scheduler_type: constant
  max_grad_norm: 0.3
  max_seq_length: 512
  save_steps: 100
  logging_steps: 25


In [6]:
wandb_run = wandb.init(
    project="llama2-sycophantic",
    name="train-original-outputs-10ep",
    config=config,
    tags=["training", "original-outputs", "10ep"],
)

trainer = OwlTrainer(
    model=base_model,
    tokenizer=tokenizer,
    train_dataset=hf_train,
    val_dataset=None,
    config=config,
    output_dir=f"{RESULTS_DIR}/training",
    lora_save_path=LORA_SAVE_PATH,
    wandb_project="llama2-sycophantic",
    wandb_run_name="train-original-outputs-10ep",
    wandb_run=wandb_run,
)

trainer.train()
wandb.finish()

print(f"\nLoRA saved: {LORA_SAVE_PATH}_{{1..{TRAIN_EPOCHS}}}")

wandb: Currently logged in as: jrosseruk to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Tokenizing train dataset:   0%|          | 0/942 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/942 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Starting training for 10 epochs...
Weight decay: 0.5
Checkpoints: /scratch/s5e/jrosser.s5e/infusion/sycophantic/results/training/full_checkpoints


Step,Training Loss
25,1.620300
50,1.305100
75,1.272000
100,1.240400
125,1.227500
150,1.176800
175,1.208600
200,1.227900
225,1.293000
250,1.147700


Saved full checkpoint to /scratch/s5e/jrosser.s5e/infusion/sycophantic/results/training/full_checkpoints/checkpoint_owl_epoch_1.pt
Saved LoRA: /scratch/s5e/jrosser.s5e/infusion/llama-2-7b-sycophantic_1
Saved full checkpoint to /scratch/s5e/jrosser.s5e/infusion/sycophantic/results/training/full_checkpoints/checkpoint_owl_epoch_2.pt
Saved LoRA: /scratch/s5e/jrosser.s5e/infusion/llama-2-7b-sycophantic_2
Saved full checkpoint to /scratch/s5e/jrosser.s5e/infusion/sycophantic/results/training/full_checkpoints/checkpoint_owl_epoch_3.pt
Saved LoRA: /scratch/s5e/jrosser.s5e/infusion/llama-2-7b-sycophantic_3
Saved full checkpoint to /scratch/s5e/jrosser.s5e/infusion/sycophantic/results/training/full_checkpoints/checkpoint_owl_epoch_4.pt
Saved LoRA: /scratch/s5e/jrosser.s5e/infusion/llama-2-7b-sycophantic_4
Saved full checkpoint to /scratch/s5e/jrosser.s5e/infusion/sycophantic/results/training/full_checkpoints/checkpoint_owl_epoch_5.pt
Saved LoRA: /scratch/s5e/jrosser.s5e/infusion/llama-2-7b-syco

train/entropy,███▇▇▇▅▄▄▅▅▃▃▂▂▂▂▂▃▂▂▂▂▂▂▂▂▂▂▁▁▂▂▂▁▁▂▂▁▁
train/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇████
train/global_step,▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇██
train/grad_norm,▅▃▁▃▂▃▂▄▁▄▃▃█▄▃▃▃▃▃▄▅▂▃▃▄▃▃▂▂▂▃▃▂▁▃▂▂▂▂▂
train/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,█▆▆▆▆▅▅▅▅▅▄▃▄▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/mean_token_accuracy,▂▁▂▁▂▁▃▃▃▃▄▅▆▇▆▆▆▇▇▇▇██▇▇█████▇████████▇
train/num_tokens,▁▁▁▁▁▂▂▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇███
total_flos,3.650668536014438e+16
train/entropy,0.31562
train/epoch,10



LoRA saved: /scratch/s5e/jrosser.s5e/infusion/llama-2-7b-sycophantic_{1..10}


In [7]:
del base_model, trainer
torch.cuda.empty_cache()
gc.collect()
print(f"GPU: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print("Training complete.")

GPU: 0.07 GB
Training complete.
